# CRIVO — Destilação 6b v5, parte 1: o caderno do professor (A100, conta hr87676620)

**O que faz:** dois professores abertos (Apache 2.0) respondem às mesmas perguntas em português e as respostas viram dados de treino para o 46M. Plano registrado em `experimentos/destilacao_20261014/PLANO.md`.

- **professor pequeno:** `Qwen/Qwen3-1.7B` · **professor médio:** `Qwen/Qwen3-8B` (sem o modo de pensar);
- **perguntas** (`professor.py`): conversa comum e fatos da fase 5 v4, OASST2 em português, acervo e enciclopédia. Ficam de fora: quem o CRIVO é, o criador, gostos e fé (IDENTIDADE.md), crise e saúde urgente (crise.py), "não sei", nomes e lembranças, e **toda frase da régua** (dev, blinds, as 15 frases reais);
- **filtro:** português, sem falar de si como modelo, sem markdown, sem repetição, sem resposta cortada; com fatos, a regra de ouro do chat (nada de número ou nome inventado) e uso dos fatos.

**Regras (AGENTS.md):** o professor só roda aqui. Os pesos dele ficam no disco do Colab e são apagados no fim; no Drive só fica o caderno, marcado com o modelo, a revisão, a data e a licença. O notebook para se a licença não for Apache 2.0 ou MIT.

**Como rodar:** Ambiente de execução → A100 → **Executar tudo**. Leva umas 2 horas (o médio é o mais lento). Se cair, rode de novo: ele continua de onde parou. No fim, mande o print da última célula.

In [ ]:
# Célula 0 — GPU e Drive
import os, sys, json, gzip, shutil, subprocess, hashlib, random, time
from pathlib import Path
from collections import Counter, defaultdict
os.environ["HF_HOME"] = "/content/hf"          # pesos do professor: só no disco do Colab
os.environ["TOKENIZERS_PARALLELISM"] = "false"
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("ERRO: sem GPU. Ambiente de execução → Alterar tipo de ambiente → A100.")
GPU, MEM = r.stdout.strip().split(", ")
print("GPU:", GPU, MEM, "MB", flush=True)
if int(MEM) < 22000:
    raise SystemExit("ERRO: precisa de A100 (ou L4); esta GPU tem pouca memória para o professor médio.")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "dados-fase6b" / "dados_fase6b.json.gz").is_file()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei CRIVO/dados-fase6b/dados_fase6b.json.gz. Rode na conta hr87676620 (onde a 6b treinou).")
SAIDA = CRIVO / "dados-professor"
(SAIDA / "bruto").mkdir(parents=True, exist_ok=True)
print("CRIVO:", CRIVO, "| saída:", SAIDA, flush=True)

In [ ]:
# Célula 1 — código do CRIVO e o vLLM
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)
VLLM = subprocess.check_output([sys.executable, "-c", "import vllm; print(vllm.__version__)"], text=True).strip()
EXP = REPO / "experimentos" / "destilacao_20261014"
for p in (str(REPO), str(EXP)):
    if p not in sys.path:
        sys.path.insert(0, p)
for m in ("professor",):
    sys.modules.pop(m, None)
import professor as P
print("commit", COMMIT[:10], "| professor.py v", P.VERSAO, "| vllm", VLLM, flush=True)

In [ ]:
# Célula 2 — as perguntas (determinísticas; ficam no Drive)
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

PERGUNTAS = SAIDA / "perguntas.jsonl.gz"
if not PERGUNTAS.is_file():
    EXP5 = REPO / "experimentos" / "fase5_conversa_20261013"
    subprocess.run([sys.executable, str(EXP5 / "gerar_v4.py")], cwd=EXP5, check=True)
    M4 = json.loads((EXP5 / "manifesto_v4.json").read_text(encoding="utf-8"))
    F5 = {n: json.loads(gzip.open(EXP5 / f).read().decode("utf-8")) for n, f in (
        ("v1", "dados_fase5.json.gz"), ("v2", "dados_fase5_v2.json.gz"), ("v3", "dados_fase5_v3.json.gz"), ("v4", "dados_fase5_v4.json.gz"))}
    D6B = json.loads(gzip.open(CRIVO / "dados-fase6b" / "dados_fase6b.json.gz").read())
    from crivo import Crivo
    from curriculo_mundo import texto_fato
    ACERVO = [(it["nome"], [texto_fato(f) for f in it["fatos"] if f.get("natureza") != "religioso"])
              for _, it in sorted(Crivo().compositor.itens.items())]
    import enciclopedia
    ENC = enciclopedia._dados()
    if ENC is None:
        raise SystemExit("ERRO: a enciclopédia não abriu (SHA). Mande o print.")
    ps = P.montar_perguntas(F5, D6B["treino"], ACERVO, ENC)
    tmp = SAIDA / "perguntas.tmp"
    with gzip.open(tmp, "wt", encoding="utf-8") as f:
        for p in ps:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    tmp.rename(PERGUNTAS)
    (SAIDA / "perguntas_info.json").write_text(json.dumps({"n": len(ps), "fase5_v4_sha256": M4["dados_sha256"],
        "por_fonte": Counter(p["fonte"].split(":")[0] for p in ps), "commit": COMMIT, "professor_py": P.VERSAO}, indent=1))
    del F5, D6B
PERG_SHA = sha256(PERGUNTAS)
with gzip.open(PERGUNTAS, "rt", encoding="utf-8") as f:
    PS = {p["id"]: p for p in map(json.loads, f)}
print(len(PS), "perguntas | sha", PERG_SHA[:12], "|", dict(Counter(p["fonte"].split(":")[0] for p in PS.values())), flush=True)
for p in random.Random(1).sample(list(PS.values()), 4):
    print("  ", p["fonte"], "→", p["turnos"][-1]["texto"][:90], flush=True)

In [ ]:
# Célula 3 — os dois professores (cada um num processo: a memória da GPU volta inteira entre eles)
PROFESSORES = [("Qwen/Qwen3-1.7B", "pequeno"), ("Qwen/Qwen3-8B", "medio")]
for modelo, rotulo in PROFESSORES:
    t0 = time.time()
    r = subprocess.run([sys.executable, str(EXP / "gerar_professor.py"), "--modelo", modelo, "--rotulo", rotulo,
                        "--perguntas", str(PERGUNTAS), "--saida", str(SAIDA / "bruto")], cwd=EXP)
    if r.returncode != 0:
        raise SystemExit(f"ERRO no professor {rotulo}. Mande o print das últimas linhas acima.")
    print(f"== professor {rotulo} pronto em {(time.time() - t0) / 60:.0f} min", flush=True)
shutil.rmtree("/content/hf", ignore_errors=True)   # os pesos dos professores saem do Colab
print("pesos dos professores apagados do disco do Colab", flush=True)

In [ ]:
# Célula 4 — filtro, o caderno e o resumo
INFO = {r: json.loads((SAIDA / "bruto" / f"professor_{r}.json").read_text()) for _, r in PROFESSORES}
EXEMPLOS, REL = [], {}
AMOSTRA = defaultdict(list)
for _, rotulo in PROFESSORES:
    motivos, por_fonte, n = Counter(), defaultdict(Counter), 0
    for arq in sorted((SAIDA / "bruto").glob(f"bruto_{rotulo}_*.jsonl.gz")):
        with gzip.open(arq, "rt", encoding="utf-8") as f:
            for linha in f:
                b = json.loads(linha); p = PS.get(b["id"])
                if p is None:
                    continue
                n += 1
                r = P.conferir(p, b["texto"])
                if b["fim"] == "length":
                    r["ok"] = False; r["motivos"].append("cortada")
                fonte = p["fonte"].split(":")[0]
                por_fonte[fonte]["ok" if r["ok"] else "fora"] += 1
                motivos.update(r["motivos"])
                if r["ok"]:
                    EXEMPLOS.append(P.exemplo(p, r, INFO[rotulo]))
                    if len(AMOSTRA[rotulo]) < 400:
                        AMOSTRA[rotulo].append((p, r))
                elif len(AMOSTRA[rotulo + "_fora"]) < 400:
                    AMOSTRA[rotulo + "_fora"].append((p, r))
    ok = sum(c["ok"] for c in por_fonte.values())
    REL[rotulo] = {"respostas": n, "aceitas": ok, "taxa": round(ok / max(1, n), 3), "motivos": dict(motivos),
                   "por_fonte": {k: dict(v) for k, v in por_fonte.items()},
                   "regra_de_ouro": sum(e["regra_de_ouro"] for e in EXEMPLOS if e["professor"] == rotulo)}
    print(f"\n== {rotulo} ({INFO[rotulo]['modelo']}): {ok}/{n} aceitas ({100 * ok / max(1, n):.0f}%) | "
          f"com a regra de ouro: {REL[rotulo]['regra_de_ouro']}", flush=True)
    print("   fora por:", dict(motivos.most_common()), flush=True)
    print("   por fonte:", {k: dict(v) for k, v in por_fonte.items()}, flush=True)

rr = random.Random(7)
for rotulo in [r for _, r in PROFESSORES]:
    print(f"\n--- 10 aceitas do {rotulo} ---", flush=True)
    for p, r in rr.sample(AMOSTRA[rotulo], min(10, len(AMOSTRA[rotulo]))):
        print(f"[{p['fonte']}] {p['turnos'][-1]['texto'][:80]!r}\n   → {r['texto'][:260]}", flush=True)
    print(f"\n--- 4 reprovadas do {rotulo} ---", flush=True)
    for p, r in rr.sample(AMOSTRA[rotulo + "_fora"], min(4, len(AMOSTRA[rotulo + "_fora"]))):
        print(f"[{p['fonte']}] {p['turnos'][-1]['texto'][:80]!r} {r['motivos']}\n   → {r['texto'][:200]}", flush=True)

bruto = json.dumps(EXEMPLOS, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
DADOS = SAIDA / "dados_professor.json.gz"
with gzip.GzipFile(DADOS, "wb", mtime=0) as f:
    f.write(bruto)
MANIFESTO = {"protocolo": "destilacao_6b_v5_parte1", "criado_em": time.strftime("%Y-%m-%dT%H:%M:%S"), "commit": COMMIT,
             "gpu": GPU, "vllm": VLLM, "professor_py": P.VERSAO, "perguntas": len(PS), "perguntas_sha256": PERG_SHA,
             "professores": INFO, "relatorio": REL, "exemplos": len(EXEMPLOS),
             "dados_sha256": hashlib.sha256(bruto).hexdigest(), "dados_gz_sha256": sha256(DADOS),
             "aviso": "dados gerados por modelos de terceiros (professor); não são falas humanas (AGENTS.md)"}
(SAIDA / "manifesto_professor.json").write_text(json.dumps(MANIFESTO, ensure_ascii=False, indent=1))
print("\n" + json.dumps({k: MANIFESTO[k] for k in ("exemplos", "dados_sha256", "perguntas", "vllm")}, indent=1), flush=True)
print({r: {k: v for k, v in REL[r].items() if k in ("respostas", "aceitas", "taxa", "regra_de_ouro")} for r in REL}, flush=True)
print("PRONTO. Mande o print desta célula para o assistente.", flush=True)